Cria uma tabela de controle para verificar se o job ja foi executado no mês:

Primeira tarefa do Job:
- Faz o download do arquivo no formato "zip" da Receite Federal, descompacta os arquivos "csv" e grava no volume da camada Bronze.

In [0]:
import os
import zipfile
import shutil
import datetime as dt

# Definição do caminho do volume no Databricks
ano_mes_pasta = dt.datetime.now().strftime("%Y_%m")
dbutils.jobs.taskValues.set(key="ano_mes_pasta", value=ano_mes_pasta)
volume_path = f"/Volumes/databricks_cnpj_data_lakehouse/bronze/{ano_mes_pasta}"

# Garantir que o schema e o volume existam
spark.sql("CREATE SCHEMA IF NOT EXISTS databricks_cnpj_data_lakehouse.bronze")
spark.sql(f"CREATE VOLUME IF NOT EXISTS databricks_cnpj_data_lakehouse.bronze.`{ano_mes_pasta}`")

def extrair_zip(file_path, validar_crc=True):
    """
    Lê e extrai o conteúdo de um arquivo ZIP específico.
    """
    print(f"\n--- Processando: {os.path.basename(file_path)} ---")

    if not os.path.exists(file_path):
        print(f"Arquivo não encontrado: {file_path}")
        return

    # Obtém o nome do ZIP sem a extensão para renomear o CSV extraído
    nome_zip_sem_ext, _ = os.path.splitext(os.path.basename(file_path))

    try:
        with zipfile.ZipFile(file_path, 'r') as z:
            if validar_crc:
                corrompido = z.testzip()
                if corrompido is not None:
                    print(f"[ERRO] Arquivo corrompido: {file_path} (Problema em: {corrompido})")
                    return

            for info in z.infolist():
                if info.is_dir():
                    continue

                nome_arquivo_interno = info.filename
                nome_limpo_interno = os.path.basename(nome_arquivo_interno)

                # Ignora arquivos ocultos/temporários
                if nome_limpo_interno.startswith('.') or not nome_limpo_interno:
                    continue

                # Se o conteúdo interno for outro .zip, mantém o nome original para recursão.
                # Se for arquivo de dados, força o nome a ser igual ao do .zip pai com extensão .csv
                if nome_limpo_interno.lower().endswith('.zip'):
                    destino = f"{volume_path}/{nome_limpo_interno}"
                else:
                    destino = f"{volume_path}/{nome_zip_sem_ext}.csv"

                if os.path.exists(destino):
                    print(f"Já existe extraído, pulando: {os.path.basename(destino)}")
                    continue

                os.makedirs(os.path.dirname(destino), exist_ok=True)
                tmp_destino = destino + ".tmp"
                t0 = dt.datetime.now()

                try:
                    with z.open(nome_arquivo_interno) as fin, open(tmp_destino, "wb") as fout:
                        shutil.copyfileobj(fin, fout, length=16 * 1024 * 1024)
                    os.replace(tmp_destino, destino)
                except Exception as e:
                    if os.path.exists(tmp_destino):
                        os.remove(tmp_destino)
                    print(f"Falha ao extrair {nome_limpo_interno}: {e}")
                    raise

                dt_s = (dt.datetime.now() - t0).total_seconds()
                tam_mb = info.file_size / (1024 * 1024)
                print(f"Extraído: {os.path.basename(destino)} ({tam_mb:.1f} MB em {dt_s:.1f}s)")

                # Se houver um zip aninhado dentro do zip principal, extrai recursivamente
                if nome_limpo_interno.lower().endswith('.zip'):
                    extrair_zip(destino, validar_crc=False)

    except zipfile.BadZipFile:
        print(f"[ERRO] O arquivo {file_path} não é um ZIP válido.")

def processar_pasta_zips(diretorio):
    """
    Percorre a pasta e processa todos os arquivos .zip encontrados.
    """
    if not os.path.exists(diretorio):
        print(f"Diretório não encontrado: {diretorio}")
        return

    # Lista todos os arquivos da pasta com extensão .zip
    arquivos_zip = [
        os.path.join(diretorio, f) for f in os.listdir(diretorio) 
        if f.lower().endswith('.zip')
    ]

    if not arquivos_zip:
        print(f"Nenhum arquivo .zip encontrado em: {diretorio}")
        return

    print(f"Encontrados {len(arquivos_zip)} arquivo(s) ZIP para processamento.")
    
    for zip_path in sorted(arquivos_zip):
        extrair_zip(zip_path)

# Execução do pipeline
try:
    processar_pasta_zips(volume_path)
    print("\nProcessamento da pasta concluído com sucesso.")
except Exception as e:
    print(f"\nErro durante o processamento: {e}")
    raise